In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim

In [4]:
#import pdb

<p style='font-size:20px'>Binary Classification</p>

In [10]:
df = pd.read_csv('gene_expression.csv')
df.head()

,Gene One,Gene Two,Cancer Present
0,4.3,3.9,1
1,2.5,6.3,0
2,5.7,3.9,1
3,6.1,6.2,0
4,7.4,3.4,1


In [12]:
df['Cancer Present'].unique()

array([1, 0], dtype=int64)

In [14]:
len(df)

3000

In [16]:
X = df.drop('Cancer Present', axis=1)
y = df['Cancer Present']

In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=101)

scaler = StandardScaler()
scaled_X_train = scaler.fit_transform(X_train)
scaled_X_test = scaler.transform(X_test)

dataset_train = TensorDataset(torch.tensor(scaled_X_train,  dtype=torch.float32), torch.tensor(y_train.to_numpy(), dtype=torch.float32))
dataset_test = TensorDataset(torch.tensor(scaled_X_test,  dtype=torch.float32), torch.tensor(y_test.to_numpy(), dtype=torch.float32))

In [20]:
dataloader_train = DataLoader(dataset_train, batch_size=10, shuffle=True)

counter = 0

for batch_inputs, batch_labels in dataloader_train:
    if counter == 20:
        print('batch inputs shape:', batch_inputs.shape)
        print('batch inputs datatype:', batch_inputs.dtype)
        print('batch labels shape:', batch_labels.shape)
        print('batch labels datatype:', batch_labels.dtype)
    
    counter += 1
    
    if counter == 14:
        print(batch_inputs)
        print(batch_labels)

tensor([[-1.8292, -0.0719],
        [-1.2800, -0.3619],
        [-0.2365, -0.6519],
        [ 0.2579,  0.3341],
        [-0.4012,  0.2761],
        [-0.0717,  1.6681],
        [ 1.2465, -1.3479],
        [-0.3463,  0.3341],
        [-0.8955,  0.5661],
        [-1.5546, -1.5799]])
tensor([0., 0., 1., 0., 1., 0., 1., 1., 0., 0.])
batch inputs shape: torch.Size([10, 2])
batch inputs datatype: torch.float32
batch labels shape: torch.Size([10])
batch labels datatype: torch.float32


In [6]:
class NNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2,32)
        self.fc2 = nn.Linear(32, 16)
        self.fc3 = nn.Linear(16, 1)
        self.r = nn.ReLU()
        self.s = nn.Sigmoid()

    def forward(self, x):
        x = self.r(self.fc1(x))
        x = self.r(self.fc2(x))
        x = self.s(self.fc3(x))
        return x

In [7]:
net = NNModel()
optimizer = optim.Adam(net.parameters(), lr=0.001)
criterion = nn.BCELoss()

In [60]:
num_epochs = 500

for i in range(num_epochs):
    for features, target in dataloader_train:

        target = target.unsqueeze(1)
        
        optimizer.zero_grad()        # otherwise the gradients keep being added up across batches 
        prediction = net(features) 
        loss = criterion(prediction, target)    
        loss.backward()
        optimizer.step()

    if (i+1) % 50 == 0:
        print(f'{round((i+1)/num_epochs*100,2)} % done.')

10.0 % done.
20.0 % done.
30.0 % done.
40.0 % done.
50.0 % done.
60.0 % done.
70.0 % done.
80.0 % done.
90.0 % done.
100.0 % done.


In [62]:
dataloader_test = DataLoader(dataset_test, batch_size=len(dataset_test), shuffle=False)
net.eval()

correct_predictions = 0
predictions = 0

with torch.no_grad():
    for features, targets in dataloader_test:
        
        outputs = net(features)
        
        pred_class = (outputs >= 0.5).float().squeeze()  # If output >= 0.5, class 1, otherwise class 0
        correct_predictions += (pred_class == targets).sum().item()  # Count correct predictions
        predictions += targets.size(0)  # Count total number of predictions (batch size)

acc = correct_predictions/predictions
print(f'Accuracy: {round(acc, 4)}')

# Set the model back to training mode
net.train()

Accuracy: 0.9344


NNModel(
  (fc1): Linear(in_features=2, out_features=32, bias=True)
  (fc2): Linear(in_features=32, out_features=16, bias=True)
  (fc3): Linear(in_features=16, out_features=1, bias=True)
  (r): ReLU()
  (s): Sigmoid()
)

In [10]:
prediction = torch.tensor([0.4, 0.7])
result = torch.tensor([1, 0])

loss = criterion(prediction.double(), result.double())
loss

tensor(1.0601, dtype=torch.float64)

In [11]:
1/(1+np.exp(-0.4)), 1/(1+np.exp(-0.7))

(0.598687660112452, 0.6681877721681662)

In [13]:
-np.log(0.598687660112452)-np.log(1-0.6681877721681662)

1.6162013012854106

In [14]:
# BCE expects probabilities, not logits!
(-np.log(0.4) - np.log(1-0.7))/2

1.0601317681000455

In [15]:
criterion_2 = nn.BCEWithLogitsLoss()
loss_2 = criterion_2(prediction.double(), result.double())
loss_2

tensor(0.8081, dtype=torch.float64)

In [16]:
(-np.log(0.598687660112452)-np.log(1-0.6681877721681662))/2

0.8081006506427053